# 03 — Evaluation & ROI Benchmarking

Tools, `SYSTEM_PROMPT`, and `build_agent()` are imported from the shared `agent_lib.py` module, ensuring evaluation runs use the exact same pipeline as `02b_run_agent`. This makes the **Claude Sonnet 4.6 vs. GPT-4.1** comparison fair and reproducible by construction.

## Metric Hierarchy

Metrics are ordered by how much they matter for a legal-intake agent, and the deployment decision applies them in this order:

1. **Conflict-detection recall (TPR) — safety gate.** A missed conflict is the costliest error (malpractice/ethics exposure), so recall must clear a configurable threshold (`conflict_tpr_gate`, default 0.95) before a model is eligible at all. A false flag only costs a reviewer a moment to clear, so specificity (TNR) is reported but not gated.
2. **Practice-area macro-F1 / macro-recall — primary routing quality.** Routing to the correct practice team is the decision the firm acts on. Macro-averaging over the six areas keeps the `Corporate` default/catch-all from inflating the score the way plain accuracy does.
3. **Exact LEDGAR category accuracy — diagnostic only.** With 100 imbalanced classes and free-form model output, exact match is depressed by label-string mismatch and is used for relative model comparison on the shared harness, not as a headline quality number. Reported alongside category macro-recall.
4. **Latency, cost, ROI — economics and tie-breakers.** Per-intake latency/cost and effectiveness-adjusted ROI inform deployment economics and break ties between models that clear the gate.

## Evaluation Objectives

- **Routing quality** on the held-out LEDGAR test split (`category_label` as the gold label): exact category match (diagnostic) plus practice-area macro-F1 / macro-recall (primary).
- **Conflict-detection TPR/TNR** on a constructed, labeled intake set (party names drawn from `lexpath_conflicts` as positives, verified clean names as negatives), since the LEDGAR test rows name no parties. Recall is enforced as a safety gate.
- **Latency and estimated cost** per intake, per model.
- **Model comparison** (Claude Sonnet 4.6 vs. GPT-4.1) using the same tools, prompts, and fixed-seed evaluation rows, evaluated concurrently for faster wall-clock.
- **Benchmark MLflow traces** covering five named intake scenarios on Claude and one comparative GPT-4.1 run, including a graceful out-of-scope rejection.
- **Effectiveness-adjusted ROI** that weights recoverable-capacity benefits by practice-area macro-F1 (not plain accuracy) and subtracts per-intake LLM costs.
- **Deployment decision** that applies the metric hierarchy: gate on conflict recall, then rank by area macro-F1, then break ties on cost and p90 latency.

## Evaluation Outputs

- Per-model metric table: category accuracy and category macro-recall (diagnostic); practice-area accuracy, macro-recall, and macro-F1 (primary); latency and per-intake cost.
- Conflict-detection table per model (TP/FN/TN/FP, recall/TPR, specificity/TNR, precision) with PASS/FAIL against the safety gate.
- Comparative reasoning traces in MLflow.
- ROI table and cross-model financial comparison, with effectiveness weighted by practice-area macro-F1.
- A deployment recommendation that surfaces the highest-quality model among those clearing the conflict-recall gate — or flags that none qualify for unsupervised use on conflict-bearing intake.

In [0]:
# Configure Widgets
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("vs_endpoint", "lexpath_vs_endpoint", "Vector Search Endpoint")
dbutils.widgets.text("claude_endpoint", "anthropic-claude-sonnet-4-6", "Claude Serving Endpoint")
dbutils.widgets.text("gpt41_endpoint", "openai-gpt-4-1", "GPT-4.1 Serving Endpoint")
dbutils.widgets.text("n_eval", "100", "Eval rows per model") # Each row is a full agentic call, so budget ~20–60 min per model.
dbutils.widgets.text("conflict_tpr_gate", "0.95", "Min conflict recall (TPR) to pass")

In [0]:
# Cost assumptions (USD per 1M tokens) — verify against current provider pricing
# https://platform.claude.com/docs/en/about-claude/pricing
# https://platform.openai.com/docs/models/gpt-4-1
dbutils.widgets.text("claude_price_in", "3.00", "Claude $/1M input tokens")
dbutils.widgets.text("claude_price_out", "15.00", "Claude $/1M output tokens")
dbutils.widgets.text("gpt41_price_in", "2.50", "GPT-4.1 $/1M input tokens")
dbutils.widgets.text("gpt41_price_out", "10.00", "GPT-4.1 $/1M output tokens")

In [0]:
# ROI assumptions 
dbutils.widgets.text("n_attorneys", "50", "Number of attorneys")
dbutils.widgets.text("billing_rate", "300", "Avg billing rate $/hr")
dbutils.widgets.text("hours_recovered", "2", "Hours recovered /attorney/week")
dbutils.widgets.text("intakes_per_week", "40", "Estimated intakes per week")

In [0]:
# Install LangChain/Databricks/Vector Search/MLflow Stack — pinned for reproducibility
# Uninstall to ensure clean state
%pip uninstall -y langgraph langgraph-checkpoint langgraph-prebuilt langgraph-sdk
# Install fresh
%pip install --upgrade langgraph databricks-langchain databricks-vectorsearch==0.75

Found existing installation: langgraph 1.0.10
Uninstalling langgraph-1.0.10:
  Successfully uninstalled langgraph-1.0.10
Found existing installation: langgraph-checkpoint 4.1.1
Uninstalling langgraph-checkpoint-4.1.1:
  Successfully uninstalled langgraph-checkpoint-4.1.1
Found existing installation: langgraph-prebuilt 1.0.13
Uninstalling langgraph-prebuilt-1.0.13:
  Successfully uninstalled langgraph-prebuilt-1.0.13
Found existing installation: langgraph-sdk 0.3.15
Uninstalling langgraph-sdk-0.3.15:
  Successfully uninstalled langgraph-sdk-0.3.15
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Using cached langgraph-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached langgraph-1.0.10-py3-

In [0]:
# Restart Python
dbutils.library.restartPython()

In [0]:
# Import, Configure, Enable MLflow Autologging
import sys, os, time
import importlib
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), '02_agent'))  # agent_lib.py is in ../02_agent
 
import pandas as pd
import mlflow
import agent_lib
importlib.reload(agent_lib)
from pyspark.sql import functions as F
 
agent_lib.configure(
    catalog=dbutils.widgets.get("catalog"),
    schema=dbutils.widgets.get("schema"),
    vs_endpoint=dbutils.widgets.get("vs_endpoint"),
)
mlflow.langchain.autolog()
 
CLAUDE_EP = dbutils.widgets.get("claude_endpoint")
GPT41_EP = dbutils.widgets.get("gpt41_endpoint")
N_EVAL = int(dbutils.widgets.get("n_eval"))
CONFLICT_TPR_GATE = float(dbutils.widgets.get("conflict_tpr_gate"))
 
PRICES = {
    CLAUDE_EP: (float(dbutils.widgets.get("claude_price_in")), float(dbutils.widgets.get("claude_price_out"))),
    GPT41_EP: (float(dbutils.widgets.get("gpt41_price_in")), float(dbutils.widgets.get("gpt41_price_out"))),
}
 
RESULTS_TABLE = f"{agent_lib.CATALOG}.{agent_lib.SCHEMA}.lexpath_eval_results"
ROUTING_MAP = agent_lib.routing_map()

agent_lib configured — index workspace.default.ledgar_provisions_index, 100 routing labels, 10 conflict matters


In [0]:
# Build the evaluation set (held-out test split)
# Same N_EVAL rows for both models, fixed seed → like comparison.
eval_pdf = (
    spark.table(f"{agent_lib.CATALOG}.{agent_lib.SCHEMA}.ledgar_lexglue")
         .filter(F.col("split") == "test")
         .withColumn("gold_category", F.col("category_label").getItem(0))
         .select("provision_id", "provision_text", "gold_category")
         .orderBy(F.rand(seed=42))
         .limit(N_EVAL)
         .toPandas()
)
print(f"Evaluation rows: {len(eval_pdf)}")
eval_pdf.head()

Evaluation rows: 100


,provision_id,provision_text,gold_category
0,70004,"As of the Closing Date, Schedule 5.12 sets for...",Subsidiaries
1,78168,The term of this Agreement ( “ Term ” ) shall ...,Terms
2,75594,Each of the parties hereto irrevocably waives ...,Waiver Of Jury Trials
3,71125,"As soon as reasonably practicable and, in any ...",Erisa
4,72645,You agree to use your reasonable and diligent ...,Further Assurances


In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_eval(endpoint_name: str, eval_pdf: pd.DataFrame, max_workers: int = 8) -> pd.DataFrame:
    """Run evaluation on a set of legal intake provisions."""
    executor = agent_lib.build_agent(endpoint_name)
    price_in, price_out = PRICES[endpoint_name]
    
    def _one(row):
        try:
            t0 = time.time()
            out = executor.invoke({"input": row.provision_text}).get("output", "")
            latency_s = time.time() - t0
            err = None
            # Extract JSON response
            js = agent_lib.extract_json(out)
            # Handle both "category" and "predicted_category" field names
            pred_cat = (js.get("category") or js.get("predicted_category") or "").strip()
            pred_area = (ROUTING_MAP.get(pred_cat.lower()) or "").strip()
            # Estimate cost (char-based approximation; exact tokens in MLflow traces)
            in_chars = len(row.provision_text)
            out_chars = len(out)
            est_cost_usd = (in_chars / 4 * price_in + out_chars / 4 * price_out) / 1_000_000
        except Exception as e:
            latency_s, pred_cat, pred_area, est_cost_usd, err = 0.0, "", "", 0.0, str(e)[:200]
        
        return {
            "model": endpoint_name,
            "provision_id": row.provision_id,
            "gold_category": row.gold_category,
            "pred_category": pred_cat,
            "gold_area": ROUTING_MAP.get(row.gold_category.lower(), ""),
            "pred_area": pred_area,
            "correct_category": pred_cat.lower() == row.gold_category.lower(),
            "correct_area": pred_area.lower() == ROUTING_MAP.get(row.gold_category.lower(), "").lower(),
            "latency_s": latency_s,
            "est_cost_usd": est_cost_usd,
            "error": err,
        }
    
    rows = list(eval_pdf.itertuples())
    recs = [None] * len(rows)
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futs = {pool.submit(_one, r): i for i, r in enumerate(rows)}
        for fut in as_completed(futs):
            recs[futs[fut]] = fut.result()
    return pd.DataFrame(recs)

In [0]:
import numpy as np

def _macro_prf(gold: pd.Series, pred: pd.Series) -> dict:
    """Macro precision/recall/F1, one-vs-rest over classes present in gold (handles imbalance)."""
    rec, prec, f1 = [], [], []
    for c in sorted(gold.unique()):
        gpos, ppos = (gold == c), (pred == c)
        tp = int((gpos & ppos).sum()); fn = int((gpos & ~ppos).sum()); fp = int((~gpos & ppos).sum())
        r = tp / (tp + fn) if tp + fn else 0.0
        p = tp / (tp + fp) if tp + fp else 0.0
        rec.append(r); prec.append(p); f1.append(2 * p * r / (p + r) if p + r else 0.0)
    return {"macro_recall": round(float(np.mean(rec)), 3),
            "macro_precision": round(float(np.mean(prec)), 3),
            "macro_f1": round(float(np.mean(f1)), 3),
            "n_classes": len(rec)}

def summarize(results: pd.DataFrame) -> dict:
    ok = results[results.error.isna()]
    cat  = _macro_prf(ok.gold_category.str.lower(), ok.pred_category.str.lower())  # 100-way: diagnostic
    area = _macro_prf(ok.gold_area.str.lower(),     ok.pred_area.str.lower())      # 6-way: primary
    return {
        "rows": len(results),
        "errors": int(results.error.notna().sum()),
        # (3) DIAGNOSTIC — exact 100-way classification, for model-vs-model only
        "category_accuracy": round(ok.correct_category.mean(), 3),
        "category_macro_recall": cat["macro_recall"],
        # (2) PRIMARY — routing quality, imbalance-robust (don't headline plain accuracy)
        "practice_area_accuracy": round(ok.correct_area.mean(), 3),
        "area_macro_recall": area["macro_recall"],
        "area_macro_f1": area["macro_f1"],
        # (4) ECONOMICS — tie-breakers
        "mean_latency_s": round(ok.latency_s.mean(), 2),
        "p90_latency_s": round(ok.latency_s.quantile(0.9), 2),
        "mean_cost_per_intake_usd": round(ok.est_cost_usd.mean(), 5),
    }

### Run Agent: Claude Sonnet 4.6 vs GPT-4.1

In [0]:
# Claude (Wrapped in MLflow Run)
all_results = []
 
with mlflow.start_run(run_name=f"eval_claude_{N_EVAL}rows"):
    claude_results = run_eval(CLAUDE_EP, eval_pdf)
    claude_summary = summarize(claude_results)
    mlflow.log_metrics({k: v for k, v in claude_summary.items() if isinstance(v, (int, float))})
    mlflow.set_tag("model", CLAUDE_EP)
all_results.append(claude_results)
print("Claude:", claude_summary)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

[Trace(trace_id=tr-6a688a21c0a023758af7526069bea0db), Trace(trace_id=tr-c7224d7e51982f7a9e5e80baf5fd83b9), Trace(trace_id=tr-18b92834d756e1d560a737cf0bcfdb3f), Trace(trace_id=tr-a0fb2d124396a9e6067385f635f5d91f), Trace(trace_id=tr-f42dcb810f3ff19383409d8b9bd4b776), Trace(trace_id=tr-7134c85bf7239cfcc4fc915b1428763a), Trace(trace_id=tr-33abede4aafc5242887012a8fadd69e2), Trace(trace_id=tr-acfd0640d893254738d8cb1eb65e256e), Trace(trace_id=tr-7a2fbe4020d446f6a59fc0760c206efd), Trace(trace_id=tr-55033323508939844ae4ba1a45ede13e)]

In [0]:
# GPT-4.1 (Wrapped in MLflow Run)
with mlflow.start_run(run_name=f"eval_gpt41_{N_EVAL}rows"):
    gpt41_results = run_eval(GPT41_EP, eval_pdf)
    gpt41_summary = summarize(gpt41_results)
    mlflow.log_metrics({k: v for k, v in gpt41_summary.items() if isinstance(v, (int, float))})
    mlflow.set_tag("model", GPT41_EP)
all_results.append(gpt41_results)
print("GPT-4.1:", gpt41_summary)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

[Trace(trace_id=tr-dcbeeffc7d8ed24fb59bc87eef0b6019), Trace(trace_id=tr-af96044331761277208cfe0f8584934b), Trace(trace_id=tr-2616582f5b7a0b0f119dcece536e13dd), Trace(trace_id=tr-6380081ffa489067d18b8fabe072e62f), Trace(trace_id=tr-cb448f7dea3590cd81fc6990326ec711), Trace(trace_id=tr-06bc7e2168ae52399efcca470083937c), Trace(trace_id=tr-845af4e8d6f410a104108794a7814fc6), Trace(trace_id=tr-bf10ab8c413e81eb99561b79a59a585f), Trace(trace_id=tr-9bc8a8159ee6b8fddc955dd7bd1d7dde), Trace(trace_id=tr-b91fc993fb4457eafdc39b4f4ac1124b)]

In [0]:
# Per-category one-vs-rest TPR/TNR, macro-averaged over LEDGAR classes present in gold.
# TPR (recall) is the informative half under class imbalance; TNR ~1.0 with 100 classes.
import numpy as np

def macro_tpr_tnr(results: pd.DataFrame) -> dict:
    ok = results[results.error.isna()]
    gold = ok.gold_category.str.lower()
    pred = ok.pred_category.str.lower()
    tprs, tnrs = [], []
    for c in sorted(gold.unique()):          # only classes with support
        gpos, ppos = (gold == c), (pred == c)
        tp = int((gpos & ppos).sum());  fn = int((gpos & ~ppos).sum())
        tn = int((~gpos & ~ppos).sum()); fp = int((~gpos & ppos).sum())
        if tp + fn: tprs.append(tp / (tp + fn))
        if tn + fp: tnrs.append(tn / (tn + fp))
    return {
        "macro_tpr": round(float(np.mean(tprs)), 3) if tprs else None,   # = macro-recall / balanced recall
        "macro_tnr": round(float(np.mean(tnrs)), 3) if tnrs else None,
        "n_gold_classes": int(gold.nunique()),
    }

for model_name, res in [("Claude", claude_results), ("GPT-4.1", gpt41_results)]:
    print(model_name, "category macro:", macro_tpr_tnr(res))

Claude category macro: {'macro_tpr': 0.536, 'macro_tnr': 0.998, 'n_gold_classes': 52}
GPT-4.1 category macro: {'macro_tpr': 0.773, 'macro_tnr': 0.998, 'n_gold_classes': 52}


In [0]:
# Persist per-row results and show side-by-side comparison
results_df = spark.createDataFrame(pd.concat(all_results))
results_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(RESULTS_TABLE)
 
comparison = pd.DataFrame([claude_summary, gpt41_summary], index=["Claude", "GPT-4.1"])
display(spark.createDataFrame(comparison.reset_index().rename(columns={"index": "model"})))

model,rows,errors,category_accuracy,category_macro_recall,practice_area_accuracy,area_macro_recall,area_macro_f1,mean_latency_s,p90_latency_s,mean_cost_per_intake_usd
Claude,100,0,0.5,0.536,1.0,1.0,1.0,10.47,16.55,0.00461
GPT-4.1,100,0,0.81,0.773,1.0,1.0,1.0,3.74,4.31,0.00219


In [0]:
# Conflict-detection eval set — constructed, since LEDGAR test rows name no parties.
conflicts = [r.asDict() for r in spark.table(agent_lib.CONFLICTS_TABLE).collect()]

def _db_conflict(party: str) -> bool:
    """Mirror agent_lib.conflict_check matching → defines ground truth."""
    n = party.strip().lower()
    if len(n) < 3:
        return False
    return any(n in r["client_name"].lower() or n in r["opposing_party"].lower() for r in conflicts)

# Positives: every distinct stored party (client OR opposing). Negatives: fabricated clean names.
stored_names = sorted({r["client_name"] for r in conflicts} | {r["opposing_party"] for r in conflicts})
clean_names = ["Riverstone Bakery LLC", "Quentin Halloway", "Marisol Vega", "Northwind Apparel Inc",
               "Devon Asante", "Larkspur Dental Group", "Priya Raman", "Sterling Oak Vineyards",
               "Hassan Okonkwo", "Beacon Hill Tutoring"]

# Multiple phrasings per party → larger N, reduces template bias.
TEMPLATES = [
    "I need help with a contract dispute. My company had a supply agreement with {party} and they "
    "breached its terms and refuse to pay what they owe. What are my options for a claim?",
    "I'm looking to take legal action against {party} over a business deal that fell apart. They "
    "did not deliver what was promised and I've lost money as a result.",
    "{party} and I are in a disagreement over a signed agreement. I believe they are in breach and "
    "I want to understand whether I have a case.",
]

conflict_pdf = pd.DataFrame([
    {"party": p, "intake": t.format(party=p), "y_true": _db_conflict(p)}
    for p in (stored_names + clean_names)
    for t in TEMPLATES
])
print(f"Conflict eval rows: {len(conflict_pdf)} "
      f"({int(conflict_pdf.y_true.sum())} positive / {int((~conflict_pdf.y_true).sum())} negative)")

Conflict eval rows: 87 (57 positive / 30 negative)


In [0]:
# Conflict Eval Functions
def run_conflict_eval(endpoint_name: str, conflict_pdf: pd.DataFrame, max_workers: int = 8) -> pd.DataFrame:
    executor = agent_lib.build_agent(endpoint_name)

    def _one(row):
        try:
            out = executor.invoke({"input": row.intake}).get("output", "")
            err = None
        except Exception as e:
            out, err = "", str(e)[:200]
        status = (agent_lib.extract_json(out).get("conflict_status") or "").strip()
        return {"model": endpoint_name, "party": row.party, "y_true": bool(row.y_true),
                "conflict_status": status, "y_pred": status == "CONFLICT_FLAG", "error": err}

    rows = list(conflict_pdf.itertuples())
    recs = [None] * len(rows)
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futs = {pool.submit(_one, r): i for i, r in enumerate(rows)}
        for fut in as_completed(futs):
            recs[futs[fut]] = fut.result()
    return pd.DataFrame(recs)

def binary_rates(df: pd.DataFrame) -> dict:
    ok = df[df.error.isna()]
    yt, yp = ok.y_true.astype(bool).values, ok.y_pred.astype(bool).values
    tp = int((yt & yp).sum());  fn = int((yt & ~yp).sum())
    tn = int((~yt & ~yp).sum()); fp = int((~yt & yp).sum())
    return {"tp": tp, "fn": fn, "tn": tn, "fp": fp,
            "tpr_recall":      round(tp / (tp + fn), 3) if tp + fn else None,   # caught conflicts
            "tnr_specificity": round(tn / (tn + fp), 3) if tn + fp else None,   # clean intakes cleared
            "precision":       round(tp / (tp + fp), 3) if tp + fp else None,
            "errors": int(df.error.notna().sum())}

In [0]:
# Run both models, log, compare, and apply the safety gate
with mlflow.start_run(run_name="conflict_eval_claude"):
    claude_conf = run_conflict_eval(CLAUDE_EP, conflict_pdf)
    claude_conf_rates = binary_rates(claude_conf)
    mlflow.set_tag("model", CLAUDE_EP)
    mlflow.log_metrics({k: v for k, v in claude_conf_rates.items() if isinstance(v, (int, float))})

with mlflow.start_run(run_name="conflict_eval_gpt41"):
    gpt41_conf = run_conflict_eval(GPT41_EP, conflict_pdf)
    gpt41_conf_rates = binary_rates(gpt41_conf)
    mlflow.set_tag("model", GPT41_EP)
    mlflow.log_metrics({k: v for k, v in gpt41_conf_rates.items() if isinstance(v, (int, float))})

conf_cmp = pd.DataFrame([claude_conf_rates, gpt41_conf_rates], index=["Claude", "GPT-4.1"])
display(spark.createDataFrame(conf_cmp.reset_index().rename(columns={"index": "model"})))

# SAFETY GATE — a missed conflict is the costliest error, so recall must clear the threshold
print(f"\n--- Conflict safety gate (min TPR = {CONFLICT_TPR_GATE}) ---")
for name, rates in [("Claude", claude_conf_rates), ("GPT-4.1", gpt41_conf_rates)]:
    tpr = rates["tpr_recall"] or 0.0
    print(f"{name}: recall(TPR)={tpr}  specificity(TNR)={rates['tnr_specificity']}  "
          f"→ {'PASS' if tpr >= CONFLICT_TPR_GATE else 'FAIL'}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

model,tp,fn,tn,fp,tpr_recall,tnr_specificity,precision,errors
Claude,57,0,30,0,1.0,1.0,1.0,0
GPT-4.1,57,0,30,0,1.0,1.0,1.0,0



--- Conflict safety gate (min TPR = 0.95) ---
Claude: recall(TPR)=1.0  specificity(TNR)=1.0  → PASS
GPT-4.1: recall(TPR)=1.0  specificity(TNR)=1.0  → PASS


[Trace(trace_id=tr-91ccb58be00b72e4050cf497a7af4957), Trace(trace_id=tr-c9c733a7d08d1541d3221f0a80f535e5), Trace(trace_id=tr-e6d985597dc7bfe25b983b16e88c4c70), Trace(trace_id=tr-906a3772a96d850725755027bf9071fb), Trace(trace_id=tr-ba0ad52877d8b60045644f42b89837d2), Trace(trace_id=tr-fe5350a1c9ef9b1ae7abf7e7f0f9121b), Trace(trace_id=tr-38869c70ce578b93ff80b350a23dc252), Trace(trace_id=tr-5a74ce5da60ea5c6e9e32c922387bb63), Trace(trace_id=tr-d3266b11a44420e2f4ece34724e34c14), Trace(trace_id=tr-6996db2e7307dadc0c90b00b214dc84d)]

### Five Benchmark Traces

In [0]:
# 5 named scenarios → 5 MLflow traces on Claude, plus scenario 1 re-run on GPT-4.1.
benchmark_scenarios = {
    "1_arbitration_intake": "My business partner and I signed an agreement that says "
        "disputes go to arbitration, but now they filed a lawsuit in court instead. "
        "I want to enforce the arbitration clause.",
    "2_conflict_flag": "I want to sue Atlas Manufacturing. I was injured by one of their "
        "forklifts in March and they refuse to cover my medical bills. My name is Paul Vance.",
    "3_employment_intake": "My employer terminated me two weeks ago and is refusing to pay "
        "the severance spelled out in my signed offer letter.",
    "4_vague_intake": "Someone wronged me and I think I might have a case. What do I do?",
    "5_out_of_scope_rejection": "Can you just tell me whether I'd win if I represented "
        "myself? Give me your legal opinion on my chances.",
}

claude_agent = agent_lib.build_agent(CLAUDE_EP)
gpt41_agent = agent_lib.build_agent(GPT41_EP)

for name, intake in benchmark_scenarios.items():
    with mlflow.start_run(run_name=f"benchmark_{name}_claude"):
        mlflow.set_tag("model", CLAUDE_EP)
        out = claude_agent.invoke({"input": intake})
        print(f"\n=== {name} (Claude) ===\n{agent_lib.extract_json(out.get('output', ''))}")

# Comparative trace: same scenario, both models
with mlflow.start_run(run_name="benchmark_1_arbitration_intake_gpt41"):
    mlflow.set_tag("model", GPT41_EP)
    out = {}   # guard so the print below never hits an undefined name
    try:
        out = gpt41_agent.invoke({"input": benchmark_scenarios["1_arbitration_intake"]})
    except Exception as e:   # 429s via ChatDatabricks won't always be openai.RateLimitError
        print(f"GPT-4.1 benchmark call failed: {type(e).__name__}: {e}")
    print(f"\n=== 1_arbitration_intake (GPT-4.1) ===\n{agent_lib.extract_json(out.get('output', ''))}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

=== 1_arbitration_intake (Claude) ===
{'status': 'READY_FOR_REVIEW', 'issue_summary': "The client entered into a business agreement containing an arbitration clause requiring disputes to be resolved through arbitration. Despite this clause, the client's business partner has filed a lawsuit in court. The client seeks to enforce the arbitration agreement and compel the matter out of court.", 'predicted_category': 'Arbitration', 'practice_area': 'Litigation', 'parties': [], 'conflict_status': 'NOT_RUN', 'conflict_matches': [], 'clarifying_questions': [], 'routing_rationale': "Multiple retrieved LEDGAR provisions establish that disputes 'shall be settled exclusively by arbitration' under AAA rules, and that parties may apply to a court specifically to enforce arbitration provisions 

[Trace(trace_id=tr-c0e5baf184ce210787c4e39a9581afd8), Trace(trace_id=tr-665716f3942d132cd7975ebe92145b52), Trace(trace_id=tr-37a6c0941c0ecff58905c9e08927efae), Trace(trace_id=tr-fdd380e3ed7c7cfaf0cbfe88680fad83), Trace(trace_id=tr-1f2875498b604f8a0e441cb92dbf61c3), Trace(trace_id=tr-b11556e1790778984250c9d28bd0bf16)]

### ROI Model

In [0]:
# ROI Model — effectiveness-adjusted
# Business value is weighted by routing accuracy: wrong routing = zero value delivered
n_attorneys      = int(dbutils.widgets.get("n_attorneys"))
billing_rate     = float(dbutils.widgets.get("billing_rate"))
hours_recovered  = float(dbutils.widgets.get("hours_recovered"))
intakes_per_week = int(dbutils.widgets.get("intakes_per_week"))

# Max recoverable capacity if every intake routed correctly
recovered_annual_max = n_attorneys * billing_rate * hours_recovered * 52

# Effectiveness = practice-area macro-F1 (imbalance-robust; wrong routing delivers no value)
effectiveness   = {"Claude Sonnet 4.6": float(claude_summary["area_macro_f1"]),
                   "GPT-4.1":           float(gpt41_summary["area_macro_f1"])}
cost_per_intake = {"Claude Sonnet 4.6": float(claude_summary["mean_cost_per_intake_usd"]),
                   "GPT-4.1":           float(gpt41_summary["mean_cost_per_intake_usd"])}

rows = []
for model in ["Claude Sonnet 4.6", "GPT-4.1"]:
    eff         = effectiveness[model]
    cpi         = cost_per_intake[model]
    annual_cost = cpi * intakes_per_week * 52
    eff_value   = eff * recovered_annual_max
    net_benefit = eff_value - annual_cost
    roi_multiple = round(eff_value / annual_cost, 0) if annual_cost else None
    rows.append({
        "model":                      model,
        "practice_area_accuracy":     round(eff, 3),
        "cost_per_intake_usd":        round(cpi, 5),
        "annual_llm_cost_usd":        round(annual_cost, 2),
        "effective_annual_value_usd": round(eff_value, 0),
        "net_annual_benefit_usd":     round(net_benefit, 0),
        "roi_multiple":               roi_multiple,
    })

roi_df = pd.DataFrame(rows)
display(spark.createDataFrame(roi_df))

# Cross-model comparison
c = rows[0]; g = rows[1]
acc_ratio  = effectiveness["GPT-4.1"] / effectiveness["Claude Sonnet 4.6"]
cost_ratio = cost_per_intake["Claude Sonnet 4.6"] / cost_per_intake["GPT-4.1"]

print(f"\n--- Cross-Model ROI ---")
print(f"Max recoverable capacity (100% accuracy): ${recovered_annual_max:,.0f}/yr")
print(f"Claude effective value:  ${c['effective_annual_value_usd']:,.0f}/yr  |  LLM cost: ${c['annual_llm_cost_usd']:.2f}/yr  |  ROI: {c['roi_multiple']:,.0f}x")
print(f"GPT-4.1 effective value: ${g['effective_annual_value_usd']:,.0f}/yr  |  LLM cost: ${g['annual_llm_cost_usd']:.2f}/yr  |  ROI: {g['roi_multiple']:,.0f}x")
print(f"\nGPT-4.1 is {acc_ratio:.1f}x more effective at routing AND {cost_ratio:.1f}x cheaper per intake")
print(f"GPT-4.1 generates ${g['effective_annual_value_usd'] - c['effective_annual_value_usd']:,.0f}/yr more business value than Claude")

with mlflow.start_run(run_name="roi_summary"):
    for r in rows:
        tag = r['model'].replace(' ', '_').replace('.', '')
        mlflow.log_metric(f"net_benefit_{tag}",  r['net_annual_benefit_usd'])
        mlflow.log_metric(f"roi_multiple_{tag}", r['roi_multiple'])
        mlflow.log_metric(f"eff_value_{tag}",    r['effective_annual_value_usd'])
    mlflow.log_metric("recovered_capacity_max",  recovered_annual_max)
    mlflow.log_metric("gpt41_value_advantage",   g['effective_annual_value_usd'] - c['effective_annual_value_usd'])
    mlflow.log_metric("accuracy_ratio_gpt_claude", acc_ratio)
    mlflow.log_metric("cost_ratio_claude_gpt",     cost_ratio)

model,practice_area_accuracy,cost_per_intake_usd,annual_llm_cost_usd,effective_annual_value_usd,net_annual_benefit_usd,roi_multiple
Claude Sonnet 4.6,1.0,0.00461,9.59,312000.0,311990.0,32538.0
GPT-4.1,1.0,0.00219,4.56,312000.0,311995.0,68493.0



--- Cross-Model ROI ---
Max recoverable capacity (100% accuracy): $312,000/yr
Claude effective value:  $312,000/yr  |  LLM cost: $9.59/yr  |  ROI: 32,538x
GPT-4.1 effective value: $312,000/yr  |  LLM cost: $4.56/yr  |  ROI: 68,493x

GPT-4.1 is 1.0x more effective at routing AND 2.1x cheaper per intake
GPT-4.1 generates $0/yr more business value than Claude


In [0]:
# Deployment decision — apply the metric hierarchy in order
decision = pd.DataFrame([
    {"model": "Claude Sonnet 4.6",
     "conflict_tpr": claude_conf_rates["tpr_recall"] or 0.0,
     "area_macro_f1": claude_summary["area_macro_f1"],
     "cost_per_intake_usd": claude_summary["mean_cost_per_intake_usd"],
     "p90_latency_s": claude_summary["p90_latency_s"]},
    {"model": "GPT-4.1",
     "conflict_tpr": gpt41_conf_rates["tpr_recall"] or 0.0,
     "area_macro_f1": gpt41_summary["area_macro_f1"],
     "cost_per_intake_usd": gpt41_summary["mean_cost_per_intake_usd"],
     "p90_latency_s": gpt41_summary["p90_latency_s"]},
])
decision["passes_safety_gate"] = decision.conflict_tpr >= CONFLICT_TPR_GATE

# 1) gate on conflict recall  2) rank by area macro-F1  3) tie-break on cost, then p90 latency
ranked = (decision[decision.passes_safety_gate]
          .sort_values(["area_macro_f1", "cost_per_intake_usd", "p90_latency_s"],
                       ascending=[False, True, True]))
display(spark.createDataFrame(decision))

if ranked.empty:
    print(f"⚠️  No model clears the conflict-recall gate ({CONFLICT_TPR_GATE}). "
          f"Do not deploy on conflict-bearing intake without human review.")
else:
    pick = ranked.iloc[0]
    print(f"✅ Recommended: {pick.model} — passes safety gate (TPR={pick.conflict_tpr}), "
          f"highest area macro-F1 ({pick.area_macro_f1}), "
          f"${pick.cost_per_intake_usd}/intake, p90 {pick.p90_latency_s}s")

model,conflict_tpr,area_macro_f1,cost_per_intake_usd,p90_latency_s,passes_safety_gate
Claude Sonnet 4.6,1.0,1.0,0.00461,16.55,true
GPT-4.1,1.0,1.0,0.00219,4.31,true


✅ Recommended: GPT-4.1 — passes safety gate (TPR=1.0), highest area macro-F1 (1.0), $0.00219/intake, p90 4.31s


## Evaluation Summary

- **Shared pipeline**: All tools, prompts, and agent construction come from `agent_lib.py`, ensuring the evaluation environment matches the production inference path in `02b_run_agent`. Any performance differences reflect model behavior rather than implementation drift. The eval set runs concurrently (bounded thread pool) for faster wall-clock without changing per-row latency measurement.

- **Metric hierarchy**: Metrics are reported and acted on in priority order for a legal-intake agent:
  - **Conflict-detection recall (TPR) — safety gate.** A missed conflict is the costliest error, so recall must clear a configurable threshold (`conflict_tpr_gate`, default 0.95) before a model is eligible. Specificity (TNR) and precision are reported but not gated, since a false flag is cheaply cleared by a reviewer. Because LEDGAR test rows name no parties, this is measured on a constructed, labeled set (positives drawn from `lexpath_conflicts`, verified clean names as negatives).
  - **Practice-area macro-F1 / macro-recall — primary routing quality.** Macro-averaging over the six practice areas keeps the `Corporate` default/catch-all from inflating the score the way plain accuracy does; this is the headline quality metric.
  - **Exact LEDGAR category match — diagnostic only.** With 100 imbalanced classes and free-form model output, exact match is depressed by label-string mismatch, so it is used for relative model comparison on the shared harness rather than as a quality headline. Reported with category macro-recall.
  - **Latency, cost, ROI — economics and tie-breakers.**

- **Cost estimates** in the evaluation loop are character-based approximations (`chars/4`) used for rapid benchmarking. Final reporting should rely on the exact token usage captured in MLflow traces.

- **Benchmark traces**: Five Claude Sonnet 4.6 traces and one comparative GPT-4.1 trace are tagged by run name under **MLflow Experiment → Traces**, covering:
  - Standard intake
  - Conflict detection and flagging
  - Clarification requests
  - Graceful out-of-scope rejection
  - Cross-model comparison

- **ROI model**: Recoverable capacity is weighted by each model's **practice-area macro-F1** (not plain accuracy, to avoid majority-class inflation) and reduced by annual per-intake LLM costs. The notebook reports:
  - Effective annual value
  - ROI multiple
  - Cross-model dollar comparison

  Because LLM operating costs are negligible relative to recoverable capacity, ROI is primarily driven by routing quality.

- **Deployment decision**: The final recommendation applies the hierarchy in order — gate on conflict recall (TPR), then rank eligible models by practice-area macro-F1, then break ties on per-intake cost and p90 latency. If no model clears the conflict-recall gate, the notebook flags that conflict-bearing intake should not be handled without human review rather than naming a winner.